# 01 - Exploratory Data Analysis

Exploring the raw OHLCV data and engineered features for 15 S&P 500 stocks.

**Contents:**
1. Data overview and shape
2. Price history by ticker
3. Return distributions
4. Technical indicator distributions
5. Correlation heatmap
6. Target class analysis
7. Feature vs target relationships

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

from stock_signal.ingest import load_raw, to_long_format
from stock_signal.features import build_features, add_ticker_encoding

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
COLORS = plt.cm.tab20.colors
print("Imports OK")

## 1. Data Overview

In [ ]:
raw = load_raw()
long_df = to_long_format(raw)
features_df = build_features(long_df)
features_df = add_ticker_encoding(features_df)
tickers = sorted(features_df["Ticker"].unique())

print(f"Shape: {features_df.shape}")
print(f"Date range: {features_df['Date'].min().date()} to {features_df['Date'].max().date()}")
print(f"Tickers ({len(tickers)}): {tickers}")
print(f"Columns: {features_df.columns.tolist()}")

In [ ]:
features_df.describe().T

In [ ]:
print("Missing values:", features_df.isnull().sum().sum())
print("\nRows per ticker:")
print(features_df.groupby("Ticker").size().sort_values())

## 2. Price History by Ticker

In [ ]:
n_cols = 3
n_rows = (len(tickers) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    df_t = features_df[features_df["Ticker"] == ticker].sort_values("Date")
    ax = axes[i]
    ax.plot(df_t["Date"], df_t["Close"], color=COLORS[i], linewidth=1.5)
    ax.fill_between(df_t["Date"], df_t["bb_lower"], df_t["bb_upper"], alpha=0.1, color=COLORS[i])
    ax.set_title(ticker, fontweight="bold")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Closing Price with Bollinger Bands", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../results/eda_price_history.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Return Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, label in zip(axes,
        ["daily_return", "return_5d", "return_20d"],
        ["Daily Return", "5-Day Return", "20-Day Return"]):
    for i, ticker in enumerate(tickers):
        ax.hist(features_df[features_df["Ticker"]==ticker][col].dropna(),
                bins=60, alpha=0.35, color=COLORS[i], density=True)
    ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Return")
fig.suptitle("Return Distributions by Ticker", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/eda_return_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
vol = (features_df.groupby("Ticker")["daily_return"].std()
       .mul(np.sqrt(252)).sort_values(ascending=False).reset_index())
vol.columns = ["Ticker", "Ann_Vol"]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(vol["Ticker"], vol["Ann_Vol"] * 100, color=COLORS[:len(vol)])
ax.set_ylabel("Annualised Volatility (%)")
ax.set_title("Annualised Volatility by Ticker", fontweight="bold")
for bar, val in zip(bars, vol["Ann_Vol"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val*100:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("../results/eda_volatility.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Technical Indicator Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(features_df["rsi_14"].dropna(), bins=50, color="#2196F3", edgecolor="white", linewidth=0.3)
axes[0].axvline(30, color="green", linestyle="--", linewidth=1.5, label="Oversold (30)")
axes[0].axvline(70, color="red",   linestyle="--", linewidth=1.5, label="Overbought (70)")
axes[0].set_title("RSI (14)", fontweight="bold")
axes[0].legend()

axes[1].hist(features_df["macd_hist"].dropna(), bins=60, color="#9C27B0", edgecolor="white", linewidth=0.3)
axes[1].axvline(0, color="black", linewidth=1.2, linestyle="--")
axes[1].set_title("MACD Histogram", fontweight="bold")

axes[2].hist(features_df["bb_width"].dropna(), bins=60, color="#FF9800", edgecolor="white", linewidth=0.3)
axes[2].set_title("Bollinger Band Width", fontweight="bold")

fig.suptitle("Technical Indicator Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/eda_indicators.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_aapl = features_df[features_df["Ticker"] == "AAPL"].sort_values("Date")
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(df_aapl["Date"], df_aapl["Close"], color="#2196F3", linewidth=1.5)
axes[0].set_ylabel("Price ($)")
axes[0].set_title("AAPL - Price and RSI (14)", fontweight="bold")

axes[1].plot(df_aapl["Date"], df_aapl["rsi_14"], color="#9C27B0", linewidth=1.2)
axes[1].axhline(70, color="red",   linestyle="--", linewidth=1, label="Overbought (70)")
axes[1].axhline(30, color="green", linestyle="--", linewidth=1, label="Oversold (30)")
axes[1].fill_between(df_aapl["Date"], df_aapl["rsi_14"], 70,
                     where=df_aapl["rsi_14"] >= 70, alpha=0.2, color="red")
axes[1].fill_between(df_aapl["Date"], df_aapl["rsi_14"], 30,
                     where=df_aapl["rsi_14"] <= 30, alpha=0.2, color="green")
axes[1].set_ylabel("RSI (14)")
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig("../results/eda_rsi_aapl.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlation Heatmap

In [ ]:
feature_cols = ["rsi_14", "macd", "macd_signal", "macd_hist",
    "bb_width", "daily_return", "return_5d", "return_20d",
    "volatility_20d", "close_to_sma20", "close_to_sma50", "target"]

corr = features_df[feature_cols].corr()
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1, linewidths=0.5, annot_kws={"size": 9}, ax=ax)
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("../results/eda_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Target Class Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

counts = features_df["target"].value_counts().sort_index()
axes[0].bar(["No Signal (0)", "Buy Signal (1)"], counts.values,
            color=["#EF5350", "#66BB6A"], edgecolor="white")
axes[0].set_title("Overall Class Balance", fontweight="bold")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f"{v:,}\n({v/len(features_df)*100:.1f}%)", ha="center", fontsize=10)

signal_rate = (features_df.groupby("Ticker")["target"].mean()
               .sort_values(ascending=False).reset_index())
axes[1].barh(signal_rate["Ticker"], signal_rate["target"] * 100, color=COLORS[:len(signal_rate)])
axes[1].set_xlabel("Signal Rate (%)")
axes[1].set_title("Buy Signal Rate by Ticker", fontweight="bold")
axes[1].axvline(features_df["target"].mean() * 100, color="black",
                linestyle="--", linewidth=1.2, label="Mean")
axes[1].legend()

axes[2].hist(features_df[features_df["target"]==0]["future_return"]*100,
             bins=60, alpha=0.6, color="#EF5350", label="No signal", density=True)
axes[2].hist(features_df[features_df["target"]==1]["future_return"]*100,
             bins=60, alpha=0.6, color="#66BB6A", label="Buy signal", density=True)
axes[2].axvline(2, color="black", linestyle="--", linewidth=1.2, label="2% threshold")
axes[2].set_xlabel("5-Day Future Return (%)")
axes[2].set_title("Future Return by Target Class", fontweight="bold")
axes[2].legend()

fig.suptitle("Target Class Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/eda_target_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Feature vs Target Relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, label in zip(axes,
        ["rsi_14", "close_to_sma20", "volatility_20d"],
        ["RSI (14)", "Price vs SMA20", "20d Volatility"]):
    for val, color, name in [(0, "#EF5350", "No Signal"), (1, "#66BB6A", "Buy Signal")]:
        ax.hist(features_df[features_df["target"]==val][col].dropna(),
                bins=50, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(f"{label} by Signal", fontweight="bold")
    ax.legend()

fig.suptitle("Key Feature Distributions: Buy Signal vs No Signal", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/eda_feature_vs_target.png", dpi=150, bbox_inches="tight")
plt.show()

print("Mean feature values by target class:")
cols = ["rsi_14", "macd_hist", "close_to_sma20", "bb_width", "volatility_20d", "return_5d"]
print(features_df.groupby("target")[cols].mean().T.round(4))

## Summary

Key findings from EDA:

- **15 tickers** with ~450 trading days each after NaN removal from indicator warm-up periods
- **Class balance**: ~33% positive (buy signal), 67% negative — mild imbalance, handled via `scale_pos_weight`
- **Volatility** varies significantly across tickers; BA and PFE tend to be more volatile, WMT and JNJ more stable
- **RSI** is roughly normally distributed around 50; overbought/oversold conditions appear regularly
- **Correlations**: MACD components are highly correlated as expected; most indicators show low direct correlation with `target`, confirming the difficulty of the prediction task
- **Feature vs target**: Buy signal rows tend to have slightly lower RSI and closer proximity to SMA20, suggesting partial mean-reversion signal
- **Signal rate by ticker**: ranges from ~28% to ~38% — reasonably uniform across the basket

These findings motivated: XGBoost over linear models (non-linear relationships), walk-forward validation (temporal structure), and `scale_pos_weight` (class imbalance).
